In [ ]:
import warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

In [2]:
!pip install -q openpyxl
!pip install -q -U imbalanced-learn
!pip install -q tensorflow
!pip install -q 'tensorflow[and-cuda]'
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
#!pip install 'accelerate>=0.26.0'
!pip install -q torch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 --index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers==4.28.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 929.1 kB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement torchvision==0.14.1 (from versions: 0.1.6, 0.2.0, 0.15.0+cu117, 0.15.1+cu117, 0.15.2+cu117)
ERROR: No matching distribution found for torchvision==0.14.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 104.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.28.1 which is incompatible.


In [ ]:
import tensorflow as tf
print("GPU Available: ", tf.test.is_built_with_cuda())
print("GPU Devices: ", tf.config.list_physical_devices('GPU'))

In [ ]:
!nvidia-smi

Thu May  1 20:30:32 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L4                      Off | 00000000:35:00.0 Off |                    0 |
| N/A   38C    P8              16W /  72W |      3MiB / 23034MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
#this 2 packages must be in the same version

import torch, torchvision
print(f"PyTorch CUDA: {torch.version.cuda}")
print(f"torchvision CUDA: {torchvision.__version__}")

PyTorch CUDA: 11.7
torchvision CUDA: 0.14.1+cu117


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
#from datasets import Dataset

In [ ]:
tariff_df = pd.read_excel('tariff_df.xlsx')
tariff_df['snomed code'] = tariff_df['snomed code'].astype(str)
tariff_df['snomed code'] =tariff_df['snomed code'].str.replace('.0', '', regex=False)
tariff_df.head()

,tariff name,tariff type,snomed code,snomed description
0,Simple Extraction Posterior,Dental,173291009,Simple extraction of tooth (procedure)
1,Thyroid Uss,Imaging,241455000,Ultrasound scan of thyroid (procedure)
2,Carfegot Tab,Drugs,778589009,Product containing only caffeine and ergotamin...
3,Pre - Exercise Ecg,Imaging,447113005,12 lead electrocardiogram at rest (procedure)
4,Pelvic & Both Hips,Imaging,718541003,X-ray tomography of pelvis and hip (procedure)


In [ ]:
tariff_df['text'] = tariff_df['tariff name']

In [ ]:
label_encoder = LabelEncoder()
label = label_encoder.fit_transform(tariff_df['snomed code'])

In [ ]:
tariff_df['snomed code']

0        173291009
1        241455000
2        778589009
3        447113005
4        718541003
           ...    
13031    440240000
13032    444934009
13033    225133004
13034    182612008
13035    440406003
Name: snomed code, Length: 13036, dtype: object

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    tariff_df['text'].tolist(),
    label.tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
#This is ClinicalBERT - skipped this to run the new model go down first
#model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

In [ ]:
# Tokenization function

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Create datasets
train_dataset = Dataset.from_dict({
    'text': train_texts,
    'labels': train_labels
})
val_dataset = Dataset.from_dict({
    'text': val_texts,
    'labels': val_labels
})

# Tokenize datasets
examples = tariff_df
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10428 [00:00<?, ? examples/s]

Map:   0%|          | 0/2608 [00:00<?, ? examples/s]

In [ ]:
import evaluate

accuracy = evaluate.load('accuracy')

#function for metrics
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    return {"accuracy": accuracy.compute(predictions=predictions,
                                            references=labels)}

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./biobert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

#Train the model
#trainer.train()

In [ ]:
trainer.save_model('snomed_bert_model_4_30')

In [ ]:
model.save_pretrained('snomed_bert_model_4_30')
tokenizer.save_pretrained('snomed_bert_model_4_30')

('snomed_bert_model_4_30/tokenizer_config.json',
 'snomed_bert_model_4_30/special_tokens_map.json',
 'snomed_bert_model_4_30/vocab.txt',
 'snomed_bert_model_4_30/added_tokens.json',
 'snomed_bert_model_4_30/tokenizer.json')

In [ ]:
trainer.evaluate()

{'eval_loss': 5.658404350280762,
 'eval_runtime': 44.0272,
 'eval_samples_per_second': 59.236,
 'eval_steps_per_second': 3.702}

In [ ]:
#This is Snomed_BERT
model_name = "snomed_bert_model_4_30"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Put model in eval mode
model.eval()

 # Load mapping - these the unique code and their description from the training set
snomed_set = tariff_df[['snomed code', 'snomed description']].drop_duplicates()
code_to_desc = snomed_set.set_index('snomed code')['snomed description'].to_dict()

# Get the validation set ready
texts = val_dataset["text"]
true_labels = val_dataset["labels"]

predicted_labels = []
for text in texts:
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    pred_label = torch.argmax(probs, dim=1).item()
    predicted_labels.append(pred_label)

# cos0 of similarity - Convert lists to numpy arrays for cosine similarity calculation
true_labels_np = np.array(true_labels).reshape(1, -1)
predicted_labels_np = np.array(predicted_labels).reshape(1, -1)

# Calculate cosine similarity, this is for the whole set
cos_sim = cosine_similarity(true_labels_np, predicted_labels_np)[0][0]

# Create a DataFrame with text, true label, and predicted label
results_df = pd.DataFrame({
    "Text": texts,
    "True Label": label_encoder.inverse_transform(true_labels),
    "Predicted Label": label_encoder.inverse_transform(predicted_labels)
})
#Find the appropriate description for the predicted snomed code
results_df["True Label Description"] = results_df["True Label"].map(code_to_desc)
results_df["Predicted Label Description"] = results_df["Predicted Label"].map(code_to_desc)


print(f"Cosine Similarity between True and Predicted Labels: {cos_sim:.4f}")
print("\nPrediction Results:")
print(results_df)

Cosine Similarity between True and Predicted Labels: 0.8279

Prediction Results:
                                 Text  True Label Predicted Label  \
0                   Knee Joint (Both)  1290410009      1290413006   
1                  Kalbloc-10 Tablets   783048008       783048008   
2            Witzer Cefuroxime 500 mg  1137332000      1137332000   
3                     Vigafit Tablets   326715008       329968007   
4      CHARLYKING CIPROFLOXACIN 500mg   783330006       783330006   
...                               ...         ...             ...   
2603    Evans Baroque Algafen Tablets   329652003       783330006   
2604  Leg / Tibia & Fibula (Ap & Lat)   783631001        65200003   
2605                     Skigud Cream   771278006       771278006   
2606         Clarithomycin Tabs 500mg   324244008       324244008   
2607               BG Lophage Tablets   325278007       783049000   

                                 True Label Description  \
0     Plain X-ray of bilateral 

In [ ]:
import pandas as pd
import numpy as np
import torch

model.eval()

texts = val_dataset["text"]
true_labels = val_dataset["labels"]

predicted_probs = []
predicted_labels = []

for text, true_label in zip(texts, true_labels):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    predicted_label = np.argmax(probs)

    predicted_probs.append(probs)
    predicted_labels.append(predicted_label)


# cos0 of similarity - Convert lists to numpy arrays for cosine similarity calculation
true_labels_np = np.array(true_labels).reshape(1, -1)
predicted_labels_np = np.array(predicted_labels).reshape(1, -1)

# Calculate cosine similarity, this is for the whole set
cos_sim = cosine_similarity(true_labels_np, predicted_labels_np)[0][0]


# Get probability assigned to the TRUE label for each sample
confidence_in_true_class = [probs[true_label] for probs, true_label in zip(predicted_probs, true_labels)]

results_df = pd.DataFrame({
    "Text": texts,
    "True Label": label_encoder.inverse_transform(true_labels),
    "Predicted Label": label_encoder.inverse_transform(predicted_labels),
    "Confidence in True Label (%)": [f"{prob*100:.2f}%" for prob in confidence_in_true_class],
    "Correct": (np.array(true_labels) == np.array(predicted_labels)).astype(int)
})

results_df["True Label Description"] = results_df["True Label"].map(code_to_desc)
results_df["Predicted Label Description"] = results_df["Predicted Label"].map(code_to_desc)

# Print results
print(f"Cosine Similarity between True and Predicted Labels: {cos_sim:.4f}")
print(f"Overall Accuracy: {results_df['Correct'].mean():.4f}")
print("\nPrediction Results:")
print(results_df)

Cosine Similarity between True and Predicted Labels: 0.8279
Overall Accuracy: 0.2561

Prediction Results:
                                 Text  True Label Predicted Label  \
0                   Knee Joint (Both)  1290410009      1290413006   
1                  Kalbloc-10 Tablets   783048008       783048008   
2            Witzer Cefuroxime 500 mg  1137332000      1137332000   
3                     Vigafit Tablets   326715008       329968007   
4      CHARLYKING CIPROFLOXACIN 500mg   783330006       783330006   
...                               ...         ...             ...   
2603    Evans Baroque Algafen Tablets   329652003       783330006   
2604  Leg / Tibia & Fibula (Ap & Lat)   783631001        65200003   
2605                     Skigud Cream   771278006       771278006   
2606         Clarithomycin Tabs 500mg   324244008       324244008   
2607               BG Lophage Tablets   325278007       783049000   

     Confidence in True Label (%)  Correct  \
0                  

In [ ]:
results_df.head(10)

,Text,True Label,Predicted Label,Confidence in True Label (%),Correct,True Label Description,Predicted Label Description
0,Knee Joint (Both),1290410009,1290413006,0.37%,0,Plain X-ray of bilateral knee regions (procedure),"Plain X-ray of knee region, anteroposterior an..."
1,Kalbloc-10 Tablets,783048008,783048008,31.48%,1,Product containing precisely amlodipine (as am...,Product containing precisely amlodipine (as am...
2,Witzer Cefuroxime 500 mg,1137332000,1137332000,57.58%,1,Product containing precisely cefuroxime (as ce...,Product containing precisely cefuroxime (as ce...
3,Vigafit Tablets,326715008,329968007,1.21%,0,Product containing precisely sildenafil (as si...,Product containing precisely diclofenac potass...
4,CHARLYKING CIPROFLOXACIN 500mg,783330006,783330006,75.76%,1,Product containing precisely ciprofloxacin (as...,Product containing precisely ciprofloxacin (as...
5,Insulin Inj 70/30 - 100iu/Ml (Humulin Or Mixta...,784510001,777276000,0.34%,0,Product containing precisely human isophane in...,Product containing only potassium chloride and...
6,2 Hrs Post Prandial,88856000,778355009,0.47%,0,"Glucose measurement, 2 hour post prandial (pro...",Product containing only ascorbic acid in oral ...
7,Antacid Suspension - (Gestid Big ),1303964002,1303965001,3.17%,0,Product containing only aluminium hydroxide an...,Product containing aluminium hydroxide and mag...
8,High Vaginal Swab (Hvs) M/C/S,401287001,168132005,5.28%,0,"Genital microscopy, culture and sensitivities ...","Microscopy, culture and sensitivities (procedure)"
9,Telduret 40/12.5 mg Tablets,407855002,407910005,0.87%,0,Product containing precisely hydrochlorothiazi...,Product containing only artemether and lumefan...


In [ ]:
results_df.to_excel('validation.xlsx', index=True)

In [ ]:
#If we had a one item we wanted to check at a time.

def predict_new_data(text, model, tokenizer, device):

    inputs = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)


    if model.device != device:
        model.to(device)

    outputs = model(**inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(predictions, dim=-1).item()


    predicted_label = label_encoder.inverse_transform([predicted_class])[0]

    return predicted_label


import torch


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

new_text = "benilyn chesty"
prediction = predict_new_data(new_text, model, tokenizer, device)
print(f"Predicted class: {prediction}")

Predicted class: 419106007


In [ ]:
snomed_set[snomed_set['snomed code']== prediction]

,snomed code,snomed description
408,419106007,Product containing ammonium chloride and diphe...


In [5]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu117
!pip install transformers

Looking in indexes: https://download.pytorch.org/whl/cu117
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 598.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 90.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 55.8 MB/s eta 0:00:00
  Created wheel for lit: filename=lit-15.0.7-py3-none-any.whl size=89990 sha256=65c8ae740a52cb7c7ffe295af6e0adbda0b9222e36b768c17a9e9d5bcd784d6b
  Stored in directory: /root/.cache/pip/wheels/fc/5d/45/34fe9945d5

In [ ]:
#I cant see the value here yet

In [ ]:
# What is the actual predictions on validation set
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

#  metrics -- classification report and confusion
from sklearn.metrics import classification_report, confusion_matrix

# Get unique labels in validation set
unique_labels_val = np.unique(val_labels)

# Filter target names to match unique labels in validation set
target_names_val = [label_encoder.classes_[i] for i in unique_labels_val]

# Print classification report using filtered target names
print(classification_report(
    val_labels,
    preds,
    target_names=target_names_val,
    labels=unique_labels_val  # Specify labels to avoid warning
))

# Print confusion matrix
print("Confusion Matrix:")
#confusion_matrix(val_labels, preds)